# Biwenger · ¿A quién fichar hoy?

**Objetivo:** encontrar, entre el mercado y las plantillas rivales, los jugadores que pueden **sumar más puntos que los míos** o **revalorizarse** (ganar dinero), sin revisar uno a uno los cientos de jugadores de la liga.

El notebook sigue un proceso **ETL** sobre la API no oficial de Biwenger (referencia: [pablopb3/biwenger-api](https://github.com/pablopb3/biwenger-api)):

| Fase | Sección | Qué hace |
|---|---|---|
| Extracción | [1. Extracción](#1-extracción) | Descarga en directo mercado, catálogo, calendarios, reglas y plantillas y guarda una captura fechada |
| Transformación | [2. Transformación](#2-transformación) | Valida, une por ID y calcula cláusulas y bloqueos en hora de Madrid |
| Modelo | [3. Modelo de rating](#3-modelo-de-rating) | Estima puntos en los próximos 3 partidos y da un rating 0–100 |
| Resultados | [4. Resultados](#4-resultados) | **Jugadores recomendados** |

> **Ejecutar todo** actualiza los datos con tu sesión guardada cifrada. La primera vez pide tu `Authorization` de Biwenger (ver [README](README.md)); liga, manager y sistema de puntos se detectan solos. El código técnico está plegado: haz clic en la celda para desplegarlo.

## 1. Extracción

Una sola llamada descarga todos los datos y los guarda en `capturas/FECHA_UTC/`, sin sobrescribir capturas anteriores. El código está en el módulo [biwenger_datos.py](biwenger_datos.py).

| Archivo | Contenido | Uso |
|---|---|---|
| `mercado.json` | Ventas y saldo | Jugadores en venta |
| `jugadores.json` | Catálogo, precio, variación y puntos con el sistema de puntuación de tu liga | Unir todo por ID |
| `calendarios.json` | Partidos de todos los equipos con dificultad por lado | Próximos 3 rivales y localía |
| `liga.json`, `plantilla_*.json` | Clasificación y plantillas, incluida la mía | Rivales, cláusulas y comparación con mi equipo |
| `reglas_clausulas.json` | Ajustes de cláusulas y mi bloqueo para ejecutarlas | Saber cuándo se puede clausular |

<details>
<summary><b>Ver detalle: sesión cifrada, modos y seguridad</b></summary>

- **Primera ejecución:** pide en una entrada oculta el valor de la cabecera `Authorization` (`Bearer eyJ...`) copiado de la web de Biwenger (pasos en el [README](README.md)). Tras una consulta correcta se guarda cifrado **fuera del proyecto**: en Windows en `%LOCALAPPDATA%\BiwengerNotebook\sesion.dpapi` (DPAPI, solo tu usuario de Windows puede descifrarlo); en macOS/Linux en el llavero del sistema (`keyring`).
- **Detección automática:** con la sesión se consulta `/account`, que devuelve tu liga, tu manager, la competición y el sistema de puntos. Si tienes varias ligas, escribe su nombre o ID en `liga`.
- **`modo_datos = "actualizar"` (por defecto):** consulta la API. **`"local"`:** usa la última captura guardada, sin red (demostraciones).
- **HTTP 401:** la sesión ha caducado. Pon `renovar_autorizacion = True`, ejecuta y pega una nueva; después vuelve a `False`.
- **Seguridad:** la autorización se envía a curl por la entrada estándar, nunca en la línea de comandos ni en las salidas; solo se envía a `biwenger.as.com`. No se siguen redirecciones (HTTP 3xx). De la cuenta solo se guarda el bloqueo de cláusulas.
- **Integridad:** las consultas son secuenciales. Si falla alguna o una plantilla no coincide con la clasificación, no se publica una captura incompleta. Los errores puntuales del servidor (404/429/5xx) se reintentan dos veces.
- **Por qué curl:** el HTTPS nativo de Python cerraba el proceso en el equipo de desarrollo; curl viene incluido en Windows 10/11, macOS y la mayoría de Linux.
- **Dependencias:** `requirements.txt`.

</details>


In [ ]:
import biwenger_datos

liga = None                     # None = tu única liga; si tienes varias, su nombre o ID
modo_datos = "actualizar"       # "local" = última captura guardada, sin red
renovar_autorizacion = False    # True solo si aparece HTTP 401

carpeta_datos = biwenger_datos.seleccionar_captura(modo_datos, liga, renovar_autorizacion)
liga_id, mi_manager_id = biwenger_datos.identidad_captura(carpeta_datos)


## 2. Transformación

Se leen y validan los JSON de la captura y se unen **por identificador de jugador**, nunca por nombre. Los valores ausentes se muestran como «Sin datos», nunca como cero.

### 2.1. Mercado, saldo y catálogo

In [ ]:
import json
from datetime import datetime, timezone
from pathlib import Path

if carpeta_datos is None:
    raise RuntimeError("Primero selecciona o actualiza una captura completa")
ruta_mercado = carpeta_datos / "mercado.json"
with ruta_mercado.open(encoding="utf-8-sig") as archivo:
    respuesta_mercado = json.load(archivo)

if respuesta_mercado["status"] != 200:
    raise ValueError(f"La respuesta del mercado indica un error: {respuesta_mercado['status']}")

datos_mercado = respuesta_mercado["data"]
ventas = datos_mercado["sales"]
if not isinstance(ventas, list):
    raise TypeError("Se esperaba una lista de anuncios en data.sales")

saldo = datos_mercado["status"]["balance"]
limite_puja = datos_mercado["status"]["maximumBid"]

print(f"Anuncios de venta: {len(ventas)}")
print(f"Saldo: {saldo:,.0f} EUR")
print(f"Límite de puja informado (no es el saldo): {limite_puja:,.0f} EUR")

In [ ]:
ruta_jugadores = carpeta_datos / "jugadores.json"
with ruta_jugadores.open(encoding="utf-8-sig") as archivo:
    respuesta_jugadores = json.load(archivo)

if respuesta_jugadores["status"] != 200:
    raise ValueError(f"La respuesta del catálogo indica un error: {respuesta_jugadores['status']}")

datos_jugadores = respuesta_jugadores["data"]
catalogo = datos_jugadores["players"]
if not isinstance(catalogo, dict):
    raise TypeError("Se esperaba un diccionario de jugadores en data.players")

nombres_puntuacion = {
    puntuacion["id"]: puntuacion["name"]
    for puntuacion in datos_jugadores["scores"]
}
sistema_puntos = datos_jugadores["scoreID"]
if sistema_puntos not in nombres_puntuacion:
    raise ValueError(f"El catálogo no define el sistema de puntuación con id {sistema_puntos}")

jugadores_por_id = {str(player_id): jugador for player_id, jugador in catalogo.items()}
ids_sin_jugador = [
    str(venta["player"]["id"])
    for venta in ventas
    if str(venta["player"]["id"]) not in jugadores_por_id
]

print(f"Jugadores en el catálogo: {len(jugadores_por_id)}")
print(f"Sistema de puntuación: {nombres_puntuacion[sistema_puntos]}")
print(f"Anuncios sin jugador asociado: {len(ids_sin_jugador)}")
if ids_sin_jugador:
    print("IDs sin correspondencia:", ids_sin_jugador)

In [ ]:
# Funciones de formato para las tablas (no cambian ningún cálculo)
from html import escape
from IPython.display import HTML, display

def formato_euros(valor):
    return f"{valor:,.0f} €".replace(",", ".")

def formato_opcional(valor, decimales=2):
    if valor is None:
        return "Sin datos"
    return f"{valor:.{decimales}f}"

def crear_tabla_html(titulo, columnas, registros):
    def celda_html(etiqueta, indice, valor):
        alineacion = "left" if indice == 0 else "right"
        estilo = f"text-align:{alineacion};padding:8px 12px;white-space:nowrap;border-bottom:1px solid #888;"
        return f'<{etiqueta} style="{estilo}">{escape(valor)}</{etiqueta}>'

    cabecera = "".join(celda_html("th", i, valor) for i, valor in enumerate(columnas))
    cuerpo = "".join(
        "<tr>" + "".join(celda_html("td", i, valor) for i, valor in enumerate(registro)) + "</tr>"
        for registro in registros
    )
    return (
        '<div style="max-width:100%;overflow-x:auto;">'
        '<table style="border-collapse:collapse;font-variant-numeric:tabular-nums;">'
        f'<caption>{escape(titulo)}</caption><thead><tr>{cabecera}</tr></thead>'
        f'<tbody>{cuerpo}</tbody></table></div>'
    )

def desplegable(titulo, contenido_html, abierto=False):
    return (f'<details{" open" if abierto else ""}><summary style="cursor:pointer;font-weight:bold;">'
            f'{escape(titulo)}</summary>{contenido_html}</details>')

### 2.2. Plantillas rivales y cláusulas

Para cada rival: valor de mercado, cláusula, **cuándo se puede ejecutar (hora de Madrid)**, diferencia entre cláusula y valor, variación diaria y puntos. Cada equipo está en un desplegable. Mi equipo se excluye por su ID.

<details>
<summary><b>Ver detalle: cómo se calcula «Ejecutable / desbloqueo»</b></summary>

Se toma la **fecha más tardía** de cuatro bloqueos:

1. Protección del jugador (`clauseLockedUntil`).
2. Fecha de compra + días de activación de la liga (`clauseActivationDelay`, 14 días).
3. Cupo de cláusulas recibidas por el propietario (`clauseReceptionLockedUntil`).
4. Mi bloqueo general para ejecutar cláusulas (`clauseExecutionDisabledUntil`), por cupo o jornada.

El código `27` de los límites significa 2 cláusulas por periodo móvil de 7 días, tanto ejecutadas como recibidas. Fuentes: [manual oficial de cláusulas](https://www.biwenger.com/faq/clausulas-de-rescision/) y cliente web oficial v631, que también describe `priceIncrement` como la variación del valor entre ayer y hoy.

«Ejecutable» significa que no hay bloqueos en esta captura. Hay que revalidar saldo, cupos y jornada antes de operar. La diferencia entre cláusula y valor no es una ganancia garantizada.

</details>

In [ ]:
# Rivales esperados según la clasificación de la captura
clasificacion = json.loads((carpeta_datos / "liga.json").read_text(encoding="utf-8"))["data"]
if str(clasificacion["id"]) != liga_id:
    raise ValueError("La clasificación pertenece a otra liga")
rivales_esperados = {str(manager["id"]): (manager["name"], manager["teamSize"])
                     for manager in clasificacion["standings"] if str(manager["id"]) != mi_manager_id}


In [ ]:
# Plantillas rivales: cláusula, bloqueos y estado en hora de Madrid
archivos_plantillas = sorted(carpeta_datos.glob("plantilla_*.json"))
plantillas_rivales = {}
plantillas_propias_excluidas = 0
mi_plantilla_ids = []
filas_rivales = []
sin_catalogo_rivales = []

ruta_metadatos = carpeta_datos / "captura.json"
fecha_referencia = (
    datetime.fromisoformat(json.loads(ruta_metadatos.read_text(encoding="utf-8"))["fin_utc"])
    if ruta_metadatos.exists() else None
)

from zoneinfo import ZoneInfo

madrid = ZoneInfo("Europe/Madrid")
ruta_reglas = carpeta_datos / "reglas_clausulas.json"
if fecha_referencia is None or not ruta_reglas.exists():
    raise ValueError("Esta captura no incluye reglas y bloqueos globales. Ejecuta la actualización automática antes de analizar cláusulas.")
reglas_clausulas = json.loads(ruta_reglas.read_text(encoding="utf-8"))["data"]
if str(reglas_clausulas["liga_id"]) != liga_id or str(reglas_clausulas["manager_id"]) != mi_manager_id:
    raise ValueError("Las reglas guardadas no corresponden a esta liga y manager")
ajustes_clausulas = reglas_clausulas["settings"]
if ajustes_clausulas["clause"] != "steal":
    raise ValueError("El recomendador de cláusulas requiere el modo de robo; revisa la nueva configuración de la liga")

def fecha_madrid(instante):
    return instante.astimezone(madrid).strftime("%d/%m/%Y %H:%M %Z")

def marca_bloqueo(valor):
    if valor is None:
        return 0
    if not isinstance(valor, (int, float)) or isinstance(valor, bool) or valor < 0:
        raise ValueError("Fecha de bloqueo de cláusula inválida")
    datetime.fromtimestamp(valor, timezone.utc)
    return valor

def estado_clausula(propietario, referencia, recepcion=None):
    if not propietario or propietario.get("clause") is None:
        return {"estado": "Sin cláusula", "fecha": None, "fecha_objetivo": None, "motivos": []}
    if propietario.get("date") is None:
        raise ValueError("Falta fecha de adquisición para verificar la activación de una cláusula; actualiza la captura")
    activacion = marca_bloqueo(propietario["date"]) + ajustes_clausulas["clauseActivationDelay"] * 86400
    individual = max(activacion, marca_bloqueo(propietario.get("clauseLockedUntil")))
    objetivo = max(individual, marca_bloqueo(recepcion))
    ejecucion = marca_bloqueo(reglas_clausulas.get("clauseExecutionDisabledUntil"))
    bloqueos = [(individual, "Protección del jugador"),
                (marca_bloqueo(recepcion), "Cupo de cláusulas recibidas del propietario"),
                (ejecucion, "Tu bloqueo general para ejecutar cláusulas (cupo o jornada)")]
    motivos = [f"{motivo}: {fecha_madrid(datetime.fromtimestamp(tm, timezone.utc))}"
               for tm, motivo in bloqueos if tm > referencia.timestamp()]
    efectiva = datetime.fromtimestamp(max(objetivo, ejecucion), timezone.utc)
    return {"estado": fecha_madrid(efectiva) if efectiva > referencia else "Ejecutable",
            "fecha": efectiva, "fecha_objetivo": datetime.fromtimestamp(objetivo, timezone.utc), "motivos": motivos}

print("Estado de cláusulas a fecha de captura (Madrid):", fecha_madrid(fecha_referencia))
for clave, etiqueta in [("clauseExecutedLimit", "Ejecutadas"), ("clauseReceivedLimit", "Recibidas")]:
    codigo = ajustes_clausulas[clave]
    texto = "2 en cada período móvil de 7 días" if str(codigo) == "27" else f"código API {codigo}; revisar ajustes"
    print(f"{etiqueta}: {texto}")
print(f"Protección tras compra: {ajustes_clausulas['clauseActivationDelay']} días; desactivación antes de jornada: {ajustes_clausulas['clauseRoundDisabledHours']} horas (250 = día de inicio).")
print("Ejecutable significa sin bloqueos temporales informados en esta captura; revalidar saldo, cupos y jornada antes de operar.")

for ruta in archivos_plantillas:
    with ruta.open(encoding="utf-8-sig") as archivo:
        respuesta = json.load(archivo)
    if respuesta["status"] != 200:
        raise ValueError(f"Respuesta de plantilla incorrecta en {ruta.name}")
    plantilla = respuesta["data"]
    if str(plantilla["league"]["id"]) != liga_id:
        raise ValueError(f"{ruta.name} pertenece a otra liga")
    manager_id = str(plantilla["id"])
    if manager_id == mi_manager_id:
        plantillas_propias_excluidas += 1
        mi_plantilla_ids = [str(miembro["id"]) for miembro in plantilla["players"]]
        continue
    if manager_id in plantillas_rivales:
        raise ValueError(f"Hay más de un archivo para el manager {manager_id}")
    if not isinstance(plantilla["players"], list):
        raise TypeError(f"Se esperaba una lista de jugadores en {ruta.name}")
    plantillas_rivales[manager_id] = plantilla["name"]
    registros_rival = []
    for miembro in plantilla["players"]:
        jugador_id = str(miembro["id"])
        jugador = jugadores_por_id.get(jugador_id)
        if jugador is None:
            sin_catalogo_rivales.append(jugador_id)
            jugador = {}
        propietario = miembro.get("owner")
        clausula = propietario.get("clause") if propietario is not None else None
        disponibilidad = estado_clausula(propietario, fecha_referencia, plantilla.get("clauseReceptionLockedUntil"))
        valor_mercado = jugador.get("price")
        diferencia_clausula = clausula - valor_mercado if clausula is not None and valor_mercado is not None else None
        porcentaje = clausula / saldo * 100 if clausula is not None and saldo > 0 else None
        local, visitante = jugador.get("playedHome"), jugador.get("playedAway")
        partidos = local + visitante if local is not None and visitante is not None else None
        puntos = jugador.get("points")
        media = puntos / partidos if puntos is not None and partidos else None
        variacion = jugador.get("priceIncrement")
        filas_rivales.append({
            "manager_id": manager_id, "jugador_id": jugador_id,
            "clausula": clausula, "porcentaje_saldo": porcentaje,
            "valor_mercado": valor_mercado,
            "diferencia_clausula": diferencia_clausula,
            "estado_clausula": disponibilidad["estado"], "fin_bloqueo": disponibilidad["fecha"],
            "fin_bloqueo_objetivo": disponibilidad["fecha_objetivo"], "motivos_bloqueo": disponibilidad["motivos"],
            "variacion": variacion, "puntos": puntos, "partidos": partidos, "media": media,
        })
        registros_rival.append([
            jugador.get("name", f"ID {jugador_id} (sin catálogo)"),
            formato_euros(valor_mercado) if valor_mercado is not None else "Sin datos",
            formato_euros(clausula) if clausula is not None else "Sin datos",
            disponibilidad["estado"],
            (("+" if diferencia_clausula > 0 else "") + formato_euros(diferencia_clausula)) if diferencia_clausula is not None else "Sin datos",
            f"{porcentaje:.2f}%".replace(".", ",") if porcentaje is not None else "No calculable",
            (("+" if variacion > 0 else "") + formato_euros(variacion)) if variacion is not None else "Sin datos",
            formato_opcional(puntos, 0), formato_opcional(partidos, 0),
            formato_opcional(media).replace(".", ","),
        ])
    display(HTML(desplegable(f"Plantilla de {plantilla['name']} ({len(registros_rival)} jugadores)", crear_tabla_html(
        f"{plantilla['name']} — {len(registros_rival)} jugadores",
        ["Jugador", "Valor mercado (€)", "Cláusula (€)", "Ejecutable / desbloqueo (Madrid)", "Cláusula − valor mercado (€)", "% saldo (cláusula)", "Variación diaria (€/día)",
         "Puntos", "Partidos", "Pts/partido"], registros_rival,
    ))))

faltan_managers = sorted(set(rivales_esperados) - set(plantillas_rivales))
print(f"Cobertura: {len(set(plantillas_rivales) & set(rivales_esperados))}/{len(rivales_esperados)} rivales de la clasificación, {len(filas_rivales)} jugadores cargados.")
if faltan_managers:
    print("Faltan plantillas:", ", ".join(rivales_esperados[mid][0] for mid in faltan_managers))
for manager_id in plantillas_rivales:
    if manager_id in rivales_esperados:
        cantidad = sum(fila["manager_id"] == manager_id for fila in filas_rivales)
        esperado = rivales_esperados[manager_id][1]
        if cantidad != esperado:
            print(f"Tamaño distinto de la clasificación: {plantillas_rivales[manager_id]}: {cantidad} frente a {esperado}")
print(f"Archivos de plantilla propia excluidos: {plantillas_propias_excluidas}")
if not archivos_plantillas:
    print("Faltan archivos plantilla_*.json para mostrar las plantillas rivales.")
if sin_catalogo_rivales:
    print("Jugadores sin catálogo (se conservan con los datos disponibles):", sin_catalogo_rivales)

## 3. Modelo de rating

Cada jugador disponible (mercado, ventas de managers y cláusulas rivales) recibe:

- **Puntos estimados por partido** en sus **próximos 3 encuentros**, con su media en casa o fuera y la dificultad del rival.
- **Rating 0–100** = **40 % deportivo** + **50 % económico** (subida €/día) + **10 % coste** (precio frente a valor).

Después se seleccionan solo las **oportunidades**, comparándolas con **mi plantilla**.

<details>
<summary><b>Ver detalle: fórmulas</b></summary>

- **Media suavizada:** media global con 5 partidos de referencia de su posición; la media en casa o fuera se suaviza con 3 partidos de esa media global. Así, un jugador con 1 partido bueno no domina el ranking.
- **Rival:** `difficulty.rating` de la API (0–100; ≤38 fácil, ≥62 difícil). Ajuste: `±|media| × 0,004 × (50 − dificultad)`, como máximo ±20 %.
- **Deportivo:** percentil de los puntos estimados frente a los jugadores de su posición principal.
- **Económico:** media de los percentiles de la subida en €/día y de la subida en % (sobre el valor de ayer).
- **Coste:** `min(100, 50 × valor / coste de entrada)`. Pagar el valor de mercado vale 50.
- **Vía de compra:** la más barata que se puede intentar ya; si no hay ninguna, la más barata en espera. Una oferta a otro manager requiere que la acepte.

</details>

<details>
<summary><b>Ver detalle: criterios de oportunidad y posiciones múltiples</b></summary>

- **Mejora deportiva:** al menos `margen_mejora_puntos` (+0,5) pts/partido más que **mi peor jugador** en alguna de sus posiciones, y no cae más de 50.000 €/día.
- **Dinero:** sube `subida_minima_dinero` (50.000 €) o más al día. Se calcula cuántos días tardaría en recuperar el sobreprecio **al ritmo de hoy**.
- **Varias posiciones:** la API da `position` y `altPositions` (por ejemplo, Moleiro es MED/DEL). Mis jugadores cuentan en todas sus posiciones y cada candidato se compara en la posición donde más mejora. Códigos: 1 POR, 2 DEF, 3 MED, 4 DEL.
- Se incluyen jugadores bloqueados o que no alcanza el saldo; ambas cosas se indican en columnas.

</details>

### 3.1. Parámetros (mis criterios)

In [ ]:
# Rating
pesos_rating = {"deportivo": 0.40, "economico": 0.50, "coste": 0.10}
prior_partidos = 5
prior_localia = 3
ajuste_dificultad = 0.004
horizonte_rating = 3          # próximos partidos

# Oportunidades
margen_mejora_puntos = 0.5      # pts/partido por encima de tu peor jugador de la misma posición
caida_maxima_deportiva = -50000  # €/día; por debajo no se propone como mejora deportiva
subida_minima_dinero = 50000     # €/día
maximo_por_lista = 10

### 3.2. Cálculo

Celdas plegadas: proyección de puntos, rating y selección de oportunidades.

In [ ]:
# Estimación de puntos por partido (próximos 3) para todo el catálogo
from math import isfinite
from statistics import mean

if set(pesos_rating) != {"deportivo", "economico", "coste"} or any(not isfinite(p) or p < 0 for p in pesos_rating.values()) or abs(sum(pesos_rating.values()) - 1) > 1e-9:
    raise ValueError("Los tres pesos deben ser finitos, no negativos y sumar 1")
if prior_partidos < 0 or prior_localia < 0 or not 0 <= ajuste_dificultad <= 0.02 or horizonte_rating != 3:
    raise ValueError("Revisa los parámetros: priors no negativos, ajuste de dificultad entre 0 y 0,02 y horizonte de 3 partidos")

ruta_calendarios = carpeta_datos / "calendarios.json"
if not ruta_calendarios.exists():
    raise ValueError("Falta el calendario ampliado: ejecuta la actualización automática para construir el rating")
respuesta_calendarios = json.loads(ruta_calendarios.read_text(encoding="utf-8"))
if respuesta_calendarios["status"] != 200:
    raise ValueError("La captura del calendario no es válida")
calendarios_rating = respuesta_calendarios["data"]

def numero_rating(valor):
    return isinstance(valor, (int, float)) and not isinstance(valor, bool) and isfinite(valor)

def media_observada(jugador, lado=None):
    if lado is None:
        cantidades = [jugador.get("playedHome"), jugador.get("playedAway")]
        partidos = sum(cantidades) if all(numero_rating(n) and n >= 0 for n in cantidades) else None
        puntos = jugador.get("points")
    else:
        partidos, puntos = jugador.get("played" + lado), jugador.get("points" + lado)
    if not numero_rating(partidos) or partidos <= 0 or not numero_rating(puntos):
        return None, partidos
    return puntos / partidos, partidos

def percentil_rating(valor, grupo):
    if not numero_rating(valor) or not grupo:
        return None
    return 100 * (sum(x < valor for x in grupo) + 0.5 * sum(x == valor for x in grupo)) / len(grupo)

medias_rating_posicion = {}
for jugador in jugadores_por_id.values():
    media, partidos = media_observada(jugador)
    if media is not None and jugador.get("position") is not None:
        medias_rating_posicion.setdefault(jugador["position"], []).append(media)
referencias_rating = {pos: mean(valores) for pos, valores in medias_rating_posicion.items()}

def proyectar_jugador(jugador, calendarios, referencia):
    avisos, proyecciones = [], []
    media, partidos = media_observada(jugador)
    base_posicion = referencias_rating.get(jugador.get("position"))
    equipo_id = str(jugador.get("teamID"))
    if equipo_id not in calendarios:
        return [], ["Equipo sin calendario"]
    futuros, sin_fecha = [], 0
    for partido in calendarios[equipo_id]["matches"]:
        if partido.get("status") not in {"pending", "preview"}:
            continue
        if not numero_rating(partido.get("date")) or partido["date"] <= 0:
            sin_fecha += 1
            continue
        if partido["date"] > referencia.timestamp():
            futuros.append(partido)
    if sin_fecha:
        avisos.append(f"{sin_fecha} encuentros pendientes sin fecha; orden del horizonte incompleto")
    encuentros = sorted({p["id"]: p for p in futuros}.values(), key=lambda p: (p["date"], p["id"]))[:horizonte_rating]
    if len(encuentros) < horizonte_rating:
        avisos.append(f"Calendario: {len(encuentros)}/{horizonte_rating} encuentros fechados")
    if media is None or base_posicion is None:
        avisos.append("Sin media o referencia posicional suficiente para estimar puntos")
    global_suavizada = (media * partidos + prior_partidos * base_posicion) / (partidos + prior_partidos) if media is not None and base_posicion is not None else None
    for partido in encuentros:
        local = str(partido["home"]["id"]) == equipo_id
        if not local and str(partido["away"]["id"]) != equipo_id:
            raise ValueError(f"Partido {partido['id']} ajeno al equipo {equipo_id}")
        propio, rival = (partido["home"], partido["away"]) if local else (partido["away"], partido["home"])
        media_lado, n_lado = media_observada(jugador, "Home" if local else "Away")
        dificultad = (propio.get("difficulty") or {}).get("rating")
        if dificultad is not None and (not numero_rating(dificultad) or not 0 <= dificultad <= 100):
            raise ValueError(f"Dificultad inválida en partido {partido['id']}")
        if dificultad is None:
            avisos.append(f"Sin dificultad ante {rival.get('name', rival['id'])}: estimación sin ajuste de rival")
        base = global_suavizada
        if base is not None and media_lado is not None:
            base = (media_lado * n_lado + prior_localia * base) / (n_lado + prior_localia)
        elif base is not None:
            avisos.append("Sin muestra casa/fuera: se usa la media global suavizada")
        puntos = base + abs(base) * ajuste_dificultad * (50 - dificultad) if base is not None and dificultad is not None else base
        proyecciones.append({"partido_id": partido["id"], "fecha": datetime.fromtimestamp(partido["date"], timezone.utc),
            "rival": rival.get("name", str(rival["id"])), "local": local, "dificultad": dificultad,
            "media_lado": media_lado, "partidos_lado": n_lado, "base": base, "puntos": puntos})
    return proyecciones, list(dict.fromkeys(avisos))

proyecciones_rating = {}
comparables_rating = {}
subidas_rating, relativas_rating = [], []
for jugador_id, jugador in jugadores_por_id.items():
    detalle, avisos = proyectar_jugador(jugador, calendarios_rating, fecha_referencia)
    estimaciones = [p["puntos"] for p in detalle if p["puntos"] is not None]
    estimacion = mean(estimaciones) if estimaciones else None
    proyecciones_rating[jugador_id] = {"partidos": detalle, "avisos": avisos, "estimacion": estimacion}
    if estimacion is not None:
        comparables_rating.setdefault(jugador.get("position"), []).append(estimacion)
    variacion, valor = jugador.get("priceIncrement"), jugador.get("price")
    if numero_rating(variacion):
        subidas_rating.append(variacion)
        if numero_rating(valor) and valor - variacion > 0:
            relativas_rating.append(variacion / (valor - variacion))
print("Modelo: 40% deportivo, 50% económico, 10% coste. Horizonte: próximos 3 partidos fechados; sin entrenamiento ni validación predictiva.")

In [ ]:
# Rating 0–100 y vías de compra (mercado, oferta a manager, cláusula)
def puntuar_rating(jugador, coste, estimacion):
    valor, cambio = jugador.get("price"), jugador.get("priceIncrement")
    deportivo = percentil_rating(estimacion, comparables_rating.get(jugador.get("position"), []))
    relativo = cambio / (valor - cambio) if numero_rating(cambio) and numero_rating(valor) and valor - cambio > 0 else None
    absoluto_p = percentil_rating(cambio, subidas_rating)
    relativo_p = percentil_rating(relativo, relativas_rating)
    economico = mean([absoluto_p, relativo_p]) if absoluto_p is not None and relativo_p is not None else None
    coste_nota = min(100, 50 * valor / coste) if numero_rating(valor) and valor > 0 and numero_rating(coste) and coste > 0 else None
    bloques = {"deportivo": deportivo, "economico": economico, "coste": coste_nota}
    rating = sum(pesos_rating[k] * v for k, v in bloques.items()) if all(v is not None for v in bloques.values()) else None
    return rating, bloques, relativo

rutas_rating = {}
for venta in ventas:
    usuario = venta.get("user")
    if usuario is not None and str(usuario["id"]) == mi_manager_id:
        continue
    if not numero_rating(venta.get("until")) or venta["until"] <= fecha_referencia.timestamp():
        continue
    jid = str(venta["player"]["id"])
    origen = "Mercado liga" if usuario is None else plantillas_rivales.get(str(usuario["id"]), f"Manager {usuario['id']}")
    rutas_rating.setdefault(jid, []).append({"origen": origen, "via": "Puja mercado" if usuario is None else "Oferta al propietario",
        "coste": venta.get("price"), "desde": fecha_referencia, "disponibilidad": "Puja disponible" if usuario is None else "Negociable; requiere aceptación",
        "avisos": ["Precio final de puja pendiente" if usuario is None else "La oferta al manager puede ser rechazada; no equivale a cláusula ejecutable"]})
for fila in filas_rivales:
    if fila["clausula"] is None:
        continue
    rutas_rating.setdefault(fila["jugador_id"], []).append({"origen": plantillas_rivales[fila["manager_id"]], "via": "Cláusula",
        "coste": fila["clausula"], "desde": fila["fin_bloqueo"], "disponibilidad": fila["estado_clausula"],
        "avisos": fila["motivos_bloqueo"] + ["Revalidar cupos, saldo y jornada al ejecutar"]})

ranking_rating = []
for jugador_id, rutas in rutas_rating.items():
    jugador = jugadores_por_id.get(jugador_id)
    validas = [r for r in rutas if numero_rating(r["coste"]) and r["coste"] > 0]
    inmediatas = [r for r in validas if r["desde"] is not None and r["desde"] <= fecha_referencia]
    principal = min(inmediatas or validas, key=lambda r: (r["coste"], r["via"])) if validas else None
    avisos = []
    proyeccion = proyecciones_rating.get(jugador_id, {"partidos": [], "avisos": ["Sin catálogo"], "estimacion": None})
    avisos.extend(proyeccion["avisos"])
    if principal is None:
        avisos.append("Sin coste de entrada válido")
    if jugador is None:
        rating, bloques, relativo = None, {}, None
    else:
        rating, bloques, relativo = puntuar_rating(jugador, principal["coste"] if principal else None, proyeccion["estimacion"])
        if jugador.get("status") != "ok":
            avisos.append(f"Estado API: {jugador.get('status')}; revisar lesión/disponibilidad")
        if numero_rating(jugador.get("priceIncrement")) and jugador["priceIncrement"] < -50000:
            avisos.append("Caída superior a 50.000 €/día; no es un veto")
        if rating is None:
            avisos.append("Sin rating: faltan " + ", ".join(k for k, v in bloques.items() if v is None))
    detalle_partidos = []
    for partido in proyeccion["partidos"]:
        aprovechable = principal is not None and principal["desde"] is not None and partido["fecha"] > max(fecha_referencia, principal["desde"])
        detalle_partidos.append(dict(partido, aprovechable=aprovechable))
    if principal:
        avisos.extend(principal["avisos"])
    aprovechables = [p for p in detalle_partidos if p["aprovechable"] and p["puntos"] is not None]
    ranking_rating.append({"id": jugador_id, "nombre": jugador.get("name", f"ID {jugador_id} (sin catálogo)") if jugador else f"ID {jugador_id} (sin catálogo)",
        "jugador": jugador or {}, "rating": rating, "bloques": bloques, "relativo": relativo,
        "principal": principal, "rutas": rutas, "estimacion": proyeccion["estimacion"], "partidos": detalle_partidos,
        "puntos_aprovechables": sum(p["puntos"] for p in aprovechables), "n_aprovechables": len(aprovechables),
        "asequible": principal["coste"] <= saldo if principal else None, "avisos": list(dict.fromkeys(avisos))})
ranking_rating.sort(key=lambda c: (c["rating"] is None, -(c["rating"] or 0), c["nombre"], c["id"]))
print(f"Ranking completo: {len(ranking_rating)} jugadores únicos; {sum(c['rating'] is not None for c in ranking_rating)} con rating y {sum(c['rating'] is None for c in ranking_rating)} sin datos suficientes.")

In [ ]:
# Comparación con mi plantilla y selección de oportunidades
nombres_posicion = {1: "POR", 2: "DEF", 3: "MED", 4: "DEL"}

def nombre_posicion(codigo):
    return nombres_posicion.get(codigo, f"Código {codigo}")

def posiciones_jugador(jugador):
    principal = jugador.get("position")
    alternativas = jugador.get("altPositions") or []
    return list(dict.fromkeys(([principal] if principal is not None else []) + list(alternativas)))

def texto_posiciones(jugador):
    return "/".join(nombre_posicion(p) for p in posiciones_jugador(jugador)) or "Sin posición"

mi_plantilla = []
for jugador_id in mi_plantilla_ids:
    jugador = jugadores_por_id.get(jugador_id)
    if jugador is None:
        print(f"Aviso: jugador propio {jugador_id} sin catálogo; no entra en la comparación")
        continue
    mi_plantilla.append({"nombre": jugador["name"], "posiciones": posiciones_jugador(jugador),
                         "texto": texto_posiciones(jugador), "estimacion": proyecciones_rating[jugador_id]["estimacion"]})
if not mi_plantilla:
    raise ValueError("No se ha encontrado tu plantilla en la captura: no se puede comparar")

peor_propio = {}
for propio in mi_plantilla:
    if propio["estimacion"] is None:
        continue
    for posicion in propio["posiciones"]:
        actual = peor_propio.get(posicion)
        if actual is None or propio["estimacion"] < actual["estimacion"]:
            peor_propio[posicion] = propio

def mejor_sustitucion(jugador, estimacion):
    """Posición donde el candidato mejora más a tu peor jugador; None si no tienes nadie allí (hueco)."""
    opciones = []
    for posicion in posiciones_jugador(jugador):
        referencia = peor_propio.get(posicion)
        if referencia is None:
            return posicion, None, None
        opciones.append((estimacion - referencia["estimacion"], posicion, referencia))
    if not opciones:
        return None, None, None
    mejora, posicion, referencia = max(opciones, key=lambda x: x[0])
    return posicion, referencia, mejora

def proximos_rivales(candidato):
    return ", ".join(f"{p['rival']} ({'C' if p['local'] else 'F'}, {formato_opcional(p['dificultad'], 0)})"
                     for p in candidato["partidos"]) or "Sin calendario"

def cambio_euros(valor):
    return ("+" if valor > 0 else "") + formato_euros(valor)

oportunidad_deportiva, oportunidad_dinero = [], []
for candidato in ranking_rating:
    jugador, ruta = candidato["jugador"], candidato["principal"]
    variacion = jugador.get("priceIncrement")
    if candidato["rating"] is None or ruta is None or not numero_rating(variacion):
        continue
    if candidato["estimacion"] is not None and variacion >= caida_maxima_deportiva:
        posicion, referencia, mejora = mejor_sustitucion(jugador, candidato["estimacion"])
        if posicion is not None and (mejora is None or mejora >= margen_mejora_puntos):
            oportunidad_deportiva.append((candidato, posicion, referencia, mejora))
    if variacion >= subida_minima_dinero:
        oportunidad_dinero.append(candidato)

oportunidad_deportiva.sort(key=lambda x: (-(x[3] if x[3] is not None else 99), -x[0]["rating"]))
oportunidad_dinero.sort(key=lambda c: (-c["jugador"]["priceIncrement"], -c["rating"]))
ids_deportivos = {c["id"] for c, _, _, _ in oportunidad_deportiva[:maximo_por_lista]}
ids_dinero = {c["id"] for c in oportunidad_dinero[:maximo_por_lista]}

filas_deportivas = []
for orden, (candidato, posicion, referencia, mejora) in enumerate(oportunidad_deportiva[:maximo_por_lista], 1):
    jugador, ruta = candidato["jugador"], candidato["principal"]
    filas_deportivas.append([str(orden), f"{candidato['nombre']} ({texto_posiciones(jugador)})",
        formato_opcional(candidato["estimacion"]),
        f"{nombre_posicion(posicion)}: {referencia['nombre']} ({formato_opcional(referencia['estimacion'])})" if referencia
            else f"{nombre_posicion(posicion)}: hueco en tu plantilla",
        f"+{formato_opcional(mejora)}" if mejora is not None else "—",
        cambio_euros(jugador["priceIncrement"]), f"{ruta['origen']} · {ruta['via']}", formato_euros(ruta["coste"]),
        ruta["disponibilidad"], "Sí" if candidato["asequible"] else "No", formato_opcional(candidato["rating"], 1),
        proximos_rivales(candidato), "Sí" if candidato["id"] in ids_dinero else ""])

filas_dinero = []
for orden, candidato in enumerate(oportunidad_dinero[:maximo_por_lista], 1):
    jugador, ruta = candidato["jugador"], candidato["principal"]
    sobreprecio = ruta["coste"] - jugador["price"]
    dias = sobreprecio / jugador["priceIncrement"]
    filas_dinero.append([str(orden), f"{candidato['nombre']} ({texto_posiciones(jugador)})",
        cambio_euros(jugador["priceIncrement"]), f"{100 * candidato['relativo']:+.2f}%" if candidato["relativo"] is not None else "Sin datos",
        f"{ruta['origen']} · {ruta['via']}", formato_euros(ruta["coste"]), formato_euros(jugador["price"]),
        cambio_euros(sobreprecio) if sobreprecio else "0 €", "Desde el primer día" if dias <= 0 else f"{dias:.1f} días",
        ruta["disponibilidad"], "Sí" if candidato["asequible"] else "No", formato_opcional(candidato["rating"], 1),
        "Sí" if candidato["id"] in ids_deportivos else ""])


estado_texto = {"ok": "OK", "injured": "Lesionado", "doubt": "Duda", "sanctioned": "Sancionado"}
filas_mercado, compras_mercado = [], []
for candidato in ranking_rating:
    ventas_jugador = [r for r in candidato["rutas"] if r["via"] in ("Puja mercado", "Oferta al propietario")]
    if not ventas_jugador:
        continue
    venta = min(ventas_jugador, key=lambda r: (r["via"] != "Puja mercado", r["coste"] or 0))
    jugador = candidato["jugador"]
    variacion = jugador.get("priceIncrement")
    estimacion = candidato["estimacion"]
    posicion, referencia, mejora = mejor_sustitucion(jugador, estimacion) if estimacion is not None else (None, None, None)
    motivos = []
    if estimacion is not None and numero_rating(variacion) and variacion >= caida_maxima_deportiva and posicion is not None \
            and (mejora is None or mejora >= margen_mejora_puntos):
        motivos.append("Mejora tu equipo")
    if numero_rating(variacion) and variacion >= subida_minima_dinero:
        motivos.append("Para hacer dinero")
    estado = jugador.get("status")
    if motivos and estado != "ok":
        veredicto = f"Revisar estado ({estado_texto.get(estado, estado)}): " + " + ".join(motivos)
    elif motivos:
        veredicto = "✅ " + " + ".join(motivos)
        compras_mercado.append(candidato["nombre"])
    elif mejora is not None and mejora >= margen_mejora_puntos and numero_rating(variacion):
        veredicto = f"⚠️ Mejora puntos, pero baja {formato_euros(-variacion)}/día"
    elif estimacion is None:
        veredicto = "Sin datos suficientes (" + "; ".join(candidato["avisos"][:1]) + ")"
    else:
        veredicto = "No aporta"
    if referencia is not None:
        comparacion = f"{nombre_posicion(posicion)}: {referencia['nombre']} ({'+' if mejora >= 0 else ''}{formato_opcional(mejora)})"
    elif posicion is not None:
        comparacion = f"{nombre_posicion(posicion)}: hueco en tu plantilla"
    else:
        comparacion = "Sin estimación"
    prioridad = 0 if veredicto.startswith("✅") else 1 if motivos else 2 if veredicto.startswith("⚠️") else 3 if veredicto == "No aporta" else 4
    filas_mercado.append((prioridad, candidato["rating"] or -1, [
        veredicto, f"{candidato['nombre']} ({texto_posiciones(jugador)})", venta["origen"], venta["via"],
        formato_euros(venta["coste"]), formato_euros(jugador.get("price")) if numero_rating(jugador.get("price")) else "Sin datos",
        cambio_euros(variacion) if numero_rating(variacion) else "Sin datos",
        formato_opcional(estimacion), comparacion, formato_opcional(candidato["rating"], 1),
        "Sí" if venta["coste"] is not None and venta["coste"] <= saldo else "No",
        estado_texto.get(estado, estado), proximos_rivales(candidato)]))

filas_mercado.sort(key=lambda x: (x[0], -x[1]))

## 4. Resultados

Primero un **resumen** con lo esencial y, debajo, las tablas que lo justifican. El ranking completo queda en un desplegable al final.

In [ ]:
from datetime import timedelta

def lista(elementos, vacio):
    return "<ul>" + "".join(f"<li>{escape(e)}</li>" for e in elementos) + "</ul>" if elementos else f"<p><i>{escape(vacio)}</i></p>"

mercado_ok = [fila for prioridad, _, fila in filas_mercado if prioridad == 0][:3]
lineas_mercado = [f"{f[1]}: {f[3].lower()} ({f[2]}) por {f[4]} · {f[7]} pts/partido, mejora a {f[8]} · {f[6]}/día" for f in mercado_ok]
lineas_puntos = [f"{f[1]}: {f[2]} pts/partido, sustituiría a {f[3]} ({f[4]}) · {f[6]} · disponible: {f[8]}" for f in filas_deportivas[:3]]
lineas_dinero = [f"{f[1]}: {f[2]}/día · {f[4]} por {f[5]} · recupera sobreprecio: {f[8]} · disponible: {f[9]}" for f in filas_dinero[:3]]
limite_vigilancia = fecha_referencia + timedelta(hours=72)
vigilar = {}
for candidato, *_ in oportunidad_deportiva[:maximo_por_lista]:
    vigilar[candidato["id"]] = candidato
for candidato in oportunidad_dinero[:maximo_por_lista]:
    vigilar[candidato["id"]] = candidato
lineas_vigilar = [f"{c['nombre']}: se desbloquea {fecha_madrid(c['principal']['desde'])}"
                  for c in sorted(vigilar.values(), key=lambda c: c["principal"]["desde"])
                  if fecha_referencia < c["principal"]["desde"] <= limite_vigilancia]

display(HTML(
    f"<h3>Resumen · {escape(fecha_madrid(fecha_referencia))} · saldo {escape(formato_euros(saldo))}</h3>"
    "<b>🛒 Mercado de hoy: buenas compras</b>" + lista(lineas_mercado, "Hoy ningún jugador en venta cumple tus criterios.") +
    "<b>⚽ Más puntos que tu plantilla (top 3)</b>" + lista(lineas_puntos, "Ningún jugador supera el margen fijado.") +
    "<b>💰 Para hacer dinero (top 3)</b>" + lista(lineas_dinero, "Ningún jugador sube lo suficiente hoy.") +
    "<b>⏰ Se desbloquean en menos de 72 h</b>" + lista(lineas_vigilar, "Ninguna oportunidad se desbloquea en las próximas 72 h.")
))
print("Heurística sin validar: revisa lesiones y alineaciones en Biwenger antes de pujar o pagar una cláusula.")

### 4.1. Oportunidades en toda la liga

**Más puntos que mi plantilla** y **para hacer dinero**: hasta 10 jugadores por lista, del mercado y de los rivales.

In [ ]:
print("OPORTUNIDADES |", fecha_madrid(fecha_referencia), "| saldo:", formato_euros(saldo))
print("Tu plantilla (todas sus posiciones):", "; ".join(f"{p['nombre']} {p['texto']} {formato_opcional(p['estimacion'])}" for p in mi_plantilla))
print("Tu peor jugador por posición:", "; ".join(f"{nombre_posicion(pos)} {p['nombre']} {formato_opcional(p['estimacion'])}"
                                               for pos, p in sorted(peor_propio.items())))
print(f"Mejoras deportivas: {len(oportunidad_deportiva)} cumplen (se muestran {len(filas_deportivas)}). "
      f"Dinero: {len(oportunidad_dinero)} cumplen (se muestran {len(filas_dinero)}). Heurística sin validar; revisa lesiones y alineaciones.")
display(HTML(crear_tabla_html("Para ganar más puntos que tu plantilla",
    ["#", "Jugador (posiciones)", "Pts/partido estimados", "Sustituiría a (posición: jugador, pts)", "Mejora", "€/día", "Origen · vía", "Coste entrada",
     "Disponible (Madrid)", "Alcanza saldo", "Rating", "Próximos rivales (C/F, dificultad)", "También sube ≥50k"], filas_deportivas)))
display(HTML(crear_tabla_html("Para hacer dinero (subida ≥ 50.000 €/día)",
    ["#", "Jugador (posiciones)", "€/día", "% diario", "Origen · vía", "Coste entrada", "Valor mercado", "Sobreprecio",
     "Recupera sobreprecio (ritmo de hoy)", "Disponible (Madrid)", "Alcanza saldo", "Rating", "También mejora tu equipo"], filas_dinero)))

### 4.2. Mercado de hoy

Todos los jugadores en venta, con veredicto: ✅ buena compra, revisar estado, ⚠️ mejora pero baja de precio, no aporta o sin datos.

In [ ]:
print("MERCADO DE HOY |", fecha_madrid(fecha_referencia), f"| {len(filas_mercado)} jugadores en venta ajenos a tu plantilla")
if compras_mercado:
    print("Buenas compras según tus criterios:", ", ".join(compras_mercado))
else:
    print("Hoy ningún jugador en venta cumple tus criterios sin cautelas: mejor esperar o mirar las cláusulas de arriba.")
display(HTML(crear_tabla_html("Mercado de hoy evaluado con el rating",
    ["Veredicto", "Jugador (posiciones)", "Vendedor", "Vía", "Precio salida", "Valor mercado", "€/día", "Pts/partido estimados",
     "Frente a tu peor jugador (posición: jugador, mejora)", "Rating", "Alcanza saldo", "Estado API", "Próximos rivales (C/F, dificultad)"],
    [fila for _, _, fila in filas_mercado])))

### 4.3. Ranking completo (consulta)

In [ ]:
registros_rating, detalles_rating = [], []
for orden, candidato in enumerate(ranking_rating, 1):
    jugador, ruta = candidato["jugador"], candidato["principal"]
    media_casa, n_casa = media_observada(jugador, "Home")
    media_fuera, n_fuera = media_observada(jugador, "Away")
    media_global, n_total = media_observada(jugador)
    cambio = jugador.get("priceIncrement")
    cambio_texto = (("+" if cambio > 0 else "") + formato_euros(cambio)) if numero_rating(cambio) else "Sin datos"
    encuentros_texto = []
    for partido in candidato["partidos"]:
        encuentro = f"{partido['rival']} ({'casa' if partido['local'] else 'fuera'}, {partido['fecha'].astimezone(madrid):%d/%m})"
        encuentros_texto.append(encuentro)
        detalles_rating.append([candidato["nombre"], encuentro, fecha_madrid(partido["fecha"]),
            formato_opcional(partido["dificultad"], 0), formato_opcional(partido["media_lado"]), formato_opcional(partido["partidos_lado"], 0),
            formato_opcional(partido["base"]), formato_opcional(partido["puntos"]), "Sí, si fichas y participa" if partido["aprovechable"] else "No antes del desbloqueo"])
    bloques_texto = " / ".join(formato_opcional(candidato["bloques"].get(k), 1) for k in ("deportivo", "economico", "coste"))
    vias = "; ".join(f"{r['via']}: {formato_euros(r['coste']) if numero_rating(r['coste']) else 'Sin coste'} ({r['disponibilidad']})" for r in candidato["rutas"])
    registros_rating.append([str(orden), formato_opcional(candidato["rating"], 1), candidato["nombre"],
        ruta["origen"] if ruta else "Sin vía válida", ruta["via"] if ruta else "Sin datos",
        formato_euros(ruta["coste"]) if ruta else "Sin datos", cambio_texto,
        f"{100 * candidato['relativo']:+.2f}%" if candidato["relativo"] is not None else "Sin datos",
        formato_opcional(media_global), f"{formato_opcional(media_casa)} ({formato_opcional(n_casa, 0)}) / {formato_opcional(media_fuera)} ({formato_opcional(n_fuera, 0)})",
        formato_opcional(candidato["estimacion"]), bloques_texto, ruta["disponibilidad"] if ruta else "Sin datos",
        ("Sí" if candidato["asequible"] else "No") if candidato["asequible"] is not None else "Sin datos",
        f"{candidato['n_aprovechables']}/{len(candidato['partidos'])}; {candidato['puntos_aprovechables']:.2f} puntos estimados" if candidato["estimacion"] is not None else "Sin estimación",
        "; ".join(encuentros_texto) or "Sin calendario", vias, "; ".join(candidato["avisos"])])

print("Ranking completo y detalle por partido: despliega para consultarlos.")
display(HTML(desplegable(f"Ranking completo: {len(registros_rating)} jugadores (mercado y rivales, disponibles y bloqueados)",
    crear_tabla_html("Rating 0–100 | 40% puntos, 50% evolución económica, 10% coste",
    ["#", "Rating /100", "Jugador", "Origen", "Vía principal", "Coste entrada", "Variación (€/día)", "Variación diaria (%)", "Media global",
     "Media casa (n) / fuera (n)", "Pts/partido estimados (hasta 3)", "Deportivo / económico / coste", "Disponible para ti (Madrid)",
     "Alcanza saldo", "Partidos aprovechables tras desbloqueo", "Próximos rivales", "Todas las vías", "Cautelas / datos ausentes"], registros_rating))))
display(HTML(desplegable("Detalle de la estimación por partido (condicional a participar)",
    crear_tabla_html("Puntos estimados por encuentro",
    ["Jugador", "Encuentro", "Fecha Madrid", "Dificultad API /100", "Media localía", "Muestra localía", "Base suavizada", "Pts estimados", "Aprovechable para ti"], detalles_rating))))

## Limitaciones y siguientes pasos

<details>
<summary><b>Ver detalle</b></summary>

- **Lo que no sabemos:** probabilidad de jugar, minutos, titularidad, recuperación de lesiones, alineaciones ni precio final de puja o venta. `status` distinto de `ok` solo genera un aviso. Tampoco se pondera `fitness`, porque no está verificado el orden de esa lista.
- **No es un modelo entrenado:** es una regla de decisión explicable según mis preferencias. Los parámetros (pesos, suavizado, ajuste de dificultad) son heurísticos y no están calibrados.
- **Validación pendiente:** guardar las estimaciones de cada captura, compararlas con los puntos reales de esas jornadas frente a una base simple (media global), medir el error (MAE) y ajustar los parámetros solo con jornadas pasadas.
- La subida diaria puede cambiar mañana. Recuperar el sobreprecio «al ritmo de hoy» no es una garantía.

</details>